# ![Machine Learning Lab](banner.jpg)

# Laboratorio 11: Arquitectura BERT

## 1. Introducción

En este laboratorio exploraremos la arquitectura **BERT** (Bidirectional Encoder Representations from Transformers), especialmente relevante por su capacidad de crear embeddings contextuales reutilizables en tareas de PLN. Implementaremos y entrenaremos un modelo tipo BERT desde cero, luego realizaremos fine-tuning para una tarea de pregunta-respuesta, y finalmente usaremos el modelo BERT original pre-entrenado.

### Objetivos del laboratorio

1. Implementar un modelo tipo BERT desde cero
2. Realizar el pre-entrenamiento del modelo con la tarea de lenguaje enmascarado
3. Utilizar el modelo pre-entrenado para resolver tareas de PLN por medio del fine-tuning
4. Hacer uso de modelos BERT pre-entrenados disponibles en Hugging Face

### Metodología

Se utiliza un subset de [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) para el pre-entrenamiento, y el dataset [SQuAD](https://huggingface.co/datasets/rajpurkar/squad) para la tarea de pregunta-respuesta.

1. Entrenamiento del tokenizador
2. Implementación del modelo tipo BERT
3. Pre-entrenamiento del modelo
4. Preparación de datos para pregunta-respuesta
5. Fine-tuning del modelo propio
6. Fine-tuning del modelo BERT original (Hugging Face)
7. Conclusiones

## 2. Preparación del entorno

Instalamos las dependencias necesarias e importamos las librerías.

In [ ]:
!pip install sentencepiece torch matplotlib datasets scikit-learn transformers

In [ ]:
import io
import time

import matplotlib.pyplot as plt
import numpy as np
# Importación de librerías
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from transformers import BertForQuestionAnswering

# Configurar semillas para facilitar la reproducibilidad de los resultados
seed = 99
torch.manual_seed(seed)
np.random.seed(seed)

# Definición de constantes
VOCAB_SIZE = 4096 # Tamaño del vocabulario del tokenizador a entrenar
TOKENIZER_MODEL = 'sp_tokenizer.model' # Nombre del modelo del tokenizador

## 1. Entrenamiento del tokenizador

### 1.1. Dataset de pre-entrenamiento

Para entrenar el tokenizador al igual que para realizar el pre-entrenamiento del modelo, vamos a usar el dataset [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu). Este dataset consiste en 1.3 billones de tokens extraídos de páginas web con contenido educativo.

El primer paso es descargar el dataset. Para este caso vamos a descargar un subset de 100 mil artículos:

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

print("CREATING STREAM")
fw = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
print("STREAM CREATED")
dataset = []
sample_size = 100000

i = 0
for document in tqdm(fw, total=sample_size, desc="Loading dataset"):
    i += 1
    dataset.append(document['text'])
    if i >= sample_size:
      break

Veamos el contenido del primer item del dataset:

In [ ]:
print(dataset[0])

### 1.2. Entrenamiento del tokenizador

Al igual que se ha hecho en tutoriales anteriores, en este tutorial vamos a utilizar la librería de Google [sentencepiece](https://github.com/google/sentencepiece) para realizar una tokenización por subpalabras del dataset de entrenamiento.

**Nota**: La ejecución de la siguiente celda toma aproximadamente 7 minutos en Google Colab.

In [ ]:
%%time
model = io.BytesIO()
# Agregamos manualmente un nuevo token para representar a los tokens enmascarados
spm.SentencePieceTrainer.train(
    sentence_iterator=iter(dataset), model_writer=model, vocab_size=VOCAB_SIZE, pad_id=0, unk_id=1, bos_id=2, eos_id=3, user_defined_symbols='<MASK>', byte_fallback=True)
sp = spm.SentencePieceProcessor(model_proto=model.getvalue())

Guardamos el tokenizador para no tener que entrenarlo nuevamente en caso de reiniciar el notebook.

In [ ]:
with open(TOKENIZER_MODEL, 'wb') as f:
  f.write(model.getvalue())

Cargamos el tokenizador nuevamente para verificar que se guardó correctamente.

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load('sp_tokenizer.model')

Realicemos algunas pruebas del tokenizador para verificar su correcto funcionamiento:

In [ ]:
encoded = sp.encode('this is a test, <MASK>')
print(encoded)

tokens = sp.encode('this is a test, <MASK>', out_type=str)
print(tokens)

print(sp.piece_to_id('<MASK>'))

## 2. Implementación del modelo tipo BERT

En esta sección vamos a implementar un modelo tipo BERT desde cero, con una arquitectura similar, pero más pequeña que la original.

### 2.1. Preparación de los datos para el entrenamiento

Antes de implementar nuestro BERT, vamos a procesar nuestros datos de entrenamiento de modo que podamos entrenar correctamente nuestro modelo con la tarea de lenguaje enmascarado.

Inicialmente, vamos a reservar el 10% de los artículos del dataset como datos de validación:

In [ ]:
k = int(len(dataset)*0.9)
train_corpus = dataset[:k]
print(f'El set de entrenamiento tiene {len(train_corpus):,} ejemplos.')
val_corpus = dataset[k:]
print(f'El set de validación tiene {len(val_corpus):,} ejemplos.')

En segundo lugar, vamos a crear un iterador sobre el dataset, el cual se encargará de iterar sobre todos los ejemplos del dataset de forma aleatoria, y adicionalmente agrega la máscara a los tokens, y agrega un token de padding a la derecha para que todos los ejemplos sean del mismo tamaño.

In [ ]:
# Clase para iterar por lotes sobre el dataset
class CustomDataLoader:
    def __init__(self, corpus, block_size) -> None:
        self.curr_pos = 0
        self.corpus =corpus # Tokenizamos todo el corpus al inicializar CustomDataLoader
        # El corpus se itera completamente antes de iterar sobre él nuevamente
        # Sin embargo, cada vez que se recorre nuevamente se tomarán secuencias en un orden distinto para evitar el sobre ajuste del modelo
        self.examples_index = list(range(0, len(corpus))) # Se asigna un índice a cada secuencia
        self.order = np.random.permutation(self.examples_index) # Se reordenan los índices para recorrer los ejemplos en un orden distinto
        self.block_size = block_size

    # Método para obtener un lote de ejemplos
    def get_batch(self, batch_size, train=True):
        # índices de con los ejemplos del lote
        batch_examples = self.order[self.curr_pos:self.curr_pos+batch_size]
        # Puntero para recorrer el corpus completamente
        self.curr_pos += len(batch_examples)

        # Cuando se itere completamente sobre el corpus, se itera sobre él nuevamente en distinto orden
        if len(batch_examples) < batch_size:
            self.curr_pos = 0 # Se reinicia el puntero
            self.order = np.random.permutation(self.examples_index) # Se reordenan los índices para recorrer los ejemplos en un orden distinto
            # En caso de que el lote final esté vacío (no tenga ejemplos), tomar el lote nuevamente
            if  len(batch_examples) == 0:
                batch_examples = self.order[self.curr_pos:self.curr_pos+batch_size]
                self.curr_pos += len(batch_examples)

        # Cada ejemplo se trunca a la longitud del block_size
        x = sp.encode([self.corpus[idx] for idx in batch_examples])
        x = [example[:self.block_size] for example in x]
        # Se agrega padding a los ejemplos más cortos
        x = [np.pad(x_i, (0, self.block_size - len(x_i)), 'constant', constant_values=sp.pad_id()) for x_i in x]
        x = torch.tensor(x)

        # En caso de que se esté en modo de entrenamiento, se enmascaran los tokens
        if train:
            # El modelo debe predecir el 15% de los tokens
            tokens_to_predict = 0.15
            # Se generan valores aleatorios para decidir qué tokens se enmascaran
            random_vals = torch.rand(x.shape[0], self.block_size)
            predict_mask = random_vals < tokens_to_predict
            # Las etiquetas son los tokens originales de los tokens enmascarados
            y = x[predict_mask]

            # Realmente se enmascaran el 80% de los tokens a predecir
            tokens_to_mask = random_vals < tokens_to_predict*0.8
            x = x.masked_fill(tokens_to_mask, torch.tensor(sp.piece_to_id('<MASK>')))
            # Originalmente, se deja un 10% de los tokens sin enmascarar para sesgar el modelo a predecir los tokens correctos.
            # Aparte, el 10% restante se deja sin enmascarar pero se reemplaza por un token aleatorio.
            # En este caso, por simplicidad, no se reemplazan los tokens no enmascarados.

            return x, y, predict_mask

        else:
            return x

A continuación, se muestra algunos ejemplos de los datos de entrenamiento con sus respectivas labels.

In [ ]:
%%time
train_data = CustomDataLoader(train_corpus, 16)

In [ ]:
x, y, mask = train_data.get_batch(4)

print(x.shape, y.shape)

# Vamos a imprimir 4 ejemplos de secuencias de entrenamiento
mask_tokens_seen = 0
for i in range(4):
    context = [sp.id_to_piece(tok_id) for tok_id in x[i].tolist()]
    mask_tokens = mask[i].sum()
    target = [sp.id_to_piece(tok_id) for tok_id in y[mask_tokens_seen:mask_tokens_seen+mask_tokens].tolist()]
    mask_tokens_seen = mask_tokens_seen+mask_tokens
    print(f'contexto: {sp.decode(context)} -> labels de los tokens enmascarados: {target}')

In [ ]:
# Cantidad de tokens enmascarados sobre el batch
x[mask].nelement() / x.nelement()

### 2.2. Implementación de BERT

In [ ]:
# Hiper parámetros
block_size = 256 # Máximo número de tokens como input al modelo
batch_size = 128 # Tamaño del lote
model_dim = 512 # Dimensiones de los embeddings de los tokens
heads_num = 8 # Número de cabezas en el mecanismo de atención
blocks_num = 3 # Número de bloques de Transformner en el modelo

# Usar GPU si está disponible
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class SelfAttention(nn.Module):
    def __init__(self, embed_dim, head_dim):
        # embed_dim son las dimensiones del embedding de los tokens originalmente
        # head_dim son las dimensiones del vector resultante del mecanismo de atención
        super().__init__()
        self.head_dim = head_dim

        self.Wq = nn.Linear(embed_dim, head_dim, bias=False) # Matriz para generar la representación Q de los tokens
        self.Wk = nn.Linear(embed_dim, head_dim, bias=False) # Matriz para generar la representación K de los tokens
        self.Wv = nn.Linear(embed_dim, head_dim, bias=False) # Matriz para generar la representación V de los tokens

    def forward(self, x): # x.shape = [batch_size, block_size, embed_dim]
        N, T, D = x.shape
        # Crear representaciones de los tokens
        Q = self.Wq(x) # [N, T, D] @ [D, head_dim] = [N, T, head_dim]
        K = self.Wk(x) # [N, T, D] @ [D, head_dim] = [N, T, head_dim]
        V = self.Wv(x) # [N, T, D] @ [D, head_dim] = [N, T, head_dim]
        # Calcular puntajes de similitud
        att_weights = Q @ K.transpose(-1,-2) # [N, T, head_dim] @ [N, head_dim, T] = [N, T, T]
        att_weights = att_weights * self.head_dim**-0.5 # Reducir el tamaño de los puntajes de similitud

        # ÚNICOS CAMBIOS CON RESPECTO A LA IMPLEMENTACIÓN DE GPT REVISADA EN SEMANAS ANTERIORES
        # Los tokens futuros no se enmascaran, de modo que el modelo ve el contexto completo
        # masked_att = att_weights.masked_fill(self.mask[:T,:T] == 0, -torch.inf)

        att_weights = torch.nn.functional.softmax(att_weights, dim=2)
        # Resultado del mecanismo de atención
        weighted_output = att_weights @ V # [N, T, T] @ [N, T, head_dim] = [N, T, head_dim]

        return weighted_output

class MultiHeadAttention(nn.Module):
    def __init__(self, heads_num, embed_dim, head_dim) -> None:
        super().__init__()
        self.heads = nn.ModuleList([SelfAttention(embed_dim, head_dim//heads_num) for _ in range(heads_num)])
        self.dense = nn.Linear(head_dim, head_dim, bias=False)

    def forward(self, x): # x: [batch_size, block_size, emb_dim]
        heads = [h(x) for h in self.heads] # [batch_size, block_size, head_dim/heads_num] por cada elemento en la lista
        att = torch.concat(heads, dim=-1) # Se concatenan los resultados de cada cabeza para obtener [batch_size, block_size, head_dim]
        output = self.dense(att) # [batch_size, block_size, head_dim]
        return output

class FeedForward(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim) -> None:
        super().__init__()
        self.dense1 = nn.Linear(in_dim, hidden_dim)
        self.dense2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x): # x: [batch_size, block_size, head_dim]
        dense1 = F.relu(self.dense1(x)) # [batch_size, block_size, head_dim*4]
        output = self.dense2(dense1) # [batch_size, block_size, head_dim]
        return output

class Block(nn.Module):
    def __init__(self, heads_num, model_dim) -> None:
        super().__init__()
        self.attention = MultiHeadAttention(heads_num, model_dim, model_dim)
        self.ln1 = nn.LayerNorm(model_dim)
        self.ffd = FeedForward(model_dim, model_dim*4, model_dim)
        self.ln2 = nn.LayerNorm(model_dim)

        self.drop1 = nn.Dropout(0.1)
        self.drop2 = nn.Dropout(0.1)
        self.drop3 = nn.Dropout(0.1)

    def forward(self, x): # x: [batch_size, block_size, emb_dim]
        att = self.attention(x) # [batch_size, block_size, head_dim], emb_dim y head_dim deben ser iguales para que funcionen las conexiones residuales
        att = self.drop1(att)
        x = self.ln1(att + x)

        ffd = self.ffd(x) # [batch_size, block_size, head_dim]
        ffd = self.drop2(ffd)
        x = self.ln2(ffd + x)
        x = self.drop3(x)
        return x # [batch_size, block_size, head_dim]


class SimpleBERT(nn.Module):
    def __init__(self, vocab_size, model_dim, block_size, blocks_num, heads_num) -> None:
        super().__init__()
        self.E = nn.Embedding(vocab_size, model_dim)
        self.posE = nn.Embedding(block_size, model_dim) # Embedding de posición. Cada posición en el contexto (0 - block_size-1) tiene su propio embedding
        self.ln1 = nn.LayerNorm(model_dim)
        self.blocks = nn.Sequential(*[Block(heads_num, model_dim) for _ in range(blocks_num)]) # El bloque se repite el número de veces deseado
        self.dense = nn.Linear(model_dim, vocab_size, bias=False)

        # Regularización
        self.drop1 = nn.Dropout(0.1)

    def forward(self, x): # x: [batch_size, block_size]
        emb1 = self.E(x) # [batch_size, block_size, emb_dim]

        # Positional embedding
        positions = torch.arange(x.shape[1], device=device) # Se genera un número por cada token que representa su posicion en el contexto (de 0 a block_size-1)
        emb2 = self.posE(positions) # [block_size, emb_dim]

        emb = emb1 + emb2 # [batch_size, block_size, emb_dim] Se suman los embeddings
        emb = self.ln1(emb)
        emb = self.drop1(emb)

        x = self.blocks(emb) # [batch_size, block_size, head_dim]

        self.contextual_embeddings = x # Se guarda el embedding contextual para usarlos más adelante

        logits = self.dense(x) # [batch_size, block_size, vocab_size]
        return logits

model = SimpleBERT(sp.vocab_size(), model_dim, block_size, blocks_num, heads_num)
model

Si revisa la implementación de manera detallada, se podrá dar cuenta como la implementación es indentica a la arquitectura de GPT implementada en semanas anteriores. La única diferencia radica en que los tokens futuros no se enmascaran. Sin embargo, esta diferencia es importante, ya que implica que el modelo no puede ser entrenado con la tarea de predicción del token siguiente porque sería una tarea trivial para el modelo.

En su lugar, el modelo de BERT se entrena con la tarea de lenguaje enmascarado, que consiste en utilizar los tokens contextuales generado para los tokens enmascarados para predecir el token original.

<img width=600 src= />


El modelo BERT original se entrena con la tarea adicional de predecir si dos secuencias de texto son contiguas o no. En nuestra implementación, se ignora esta tarea por simplicidad y porque en investigaciones posteriores como en el paper de RoBERTa ([enlace al paper](https://arxiv.org/abs/1907.11692)) se demuestra que esta tarea se puede remover sin afectar el desempeño del modelo.

In [ ]:
print(f'Número de parametros del modelo: {sum([p.nelement() for p in model.parameters()]):,}')

## 3. Pre-entrenamiento del modelo

En esta sección vamos a realizar el entrenamiento del modelo BERT con la tarea de lenguaje enmascarado en el dataset de **FineWeb-Edu**.

In [ ]:
%time
train_data = CustomDataLoader(train_corpus, block_size)
train_eval_data = CustomDataLoader(train_corpus, block_size)
val_data = CustomDataLoader(val_corpus, block_size)

Como en tutoriales pasados, creamos una función de estimación de la función de costo para el dataset de entrenamiento y validación para hacer seguimiento al entrenamiento del modelo.

In [ ]:
# Función para estimar el valor de pérdida en el set de entrenamiento, y en el de validación
@torch.no_grad
def estimate_loss(model: torch.nn.Module, train_eval: CustomDataLoader, val_eval: CustomDataLoader, batch_size: int = 256, eval_iters: int = 10):
    model.eval() # Se debe informar a Pytorch que estamos en modo evaluación, esta información puede ser usada por alguna de las capas usadas como por ejemplo Dropout
    train_losses = []
    val_losses = []
    # Iteramos sobre un número de batches definido para tener una mejor estimación del valor de pérdida en cada uno de los sets
    for i in range(eval_iters):
        x, y, mask = train_eval.get_batch(batch_size)
        x, y, mask = x.to(device), y.to(device), mask.to(device) # Se envían los tensores al dispositivo en el que tenemos el modelo, ya sea CPU o GPU
        y_pred = model(x) # Se calculan los logits
        # Solo nos importan los logits de los tokens enmascarados
        y_pred = y_pred[mask]
        # Se calcula la función de pérdida solo para los tokens enmascarados
        loss = torch.nn.functional.cross_entropy(y_pred.view(-1, sp.vocab_size()), y.view(-1))
        train_losses.append(loss.item())

        # Se realizan los mismos pasos en el set de validación
        x, y, mask = val_eval.get_batch(batch_size)
        x, y, mask = x.to(device), y.to(device), mask.to(device)
        y_pred = model(x)
        y_pred = y_pred[mask]
        loss = torch.nn.functional.cross_entropy(y_pred.view(-1, sp.vocab_size()), y.view(-1))
        val_losses.append(loss.item())

    train_loss = sum(train_losses)/eval_iters
    val_loss = sum(val_losses)/eval_iters

    model.train()
    return train_loss, val_loss

Ahora definimos la función de entrenamiento del modelo, la cual es similar a la vista en semanas anteriores, con la diferencia de que en este caso solo estamos interesados en predecir los tokens enmascarados, por lo que solo calculamos la función de costo para estos tokens.

In [ ]:
# Función para definir el bucle de entrenamiento del modelo
def training_loop(model, lr, max_steps, data_loader: CustomDataLoader, train_eval: CustomDataLoader, val_eval: CustomDataLoader, weight_decay: float = 0.0, steps_per_log=64):
    train_losses = []
    val_losses = []

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    print(f'Entrenando por {max_steps} pasos')
    for step in range(max_steps):
        x, y, mask = data_loader.get_batch(batch_size)
        x, y, mask = x.to(device), y.to(device), mask.to(device) # Se envían los tensores al dispositivo en el que tenemos el modelo, ya sea CPU o GPU
        y_pred = model(x) # Se calculan los logits
        # Solo nos importan los logits de los tokens enmascarados
        y_pred = y_pred[mask]
        loss = torch.nn.functional.cross_entropy(y_pred.view(-1, sp.vocab_size()), y.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Logs
        if (step+1) % steps_per_log == 0 or step == 0 or step == max_steps-1:
            train_loss, valid_loss = estimate_loss(model, train_eval, val_eval, batch_size=batch_size)
            print(f'Step {step+1:4d} - Train loss: {train_loss:4.2f} - Valid loss: {valid_loss:4.2f}')
            train_losses.append(train_loss)
            val_losses.append(valid_loss)
    return train_losses, val_losses

Calculamos el número de batches necesario para recorrer el dataset de manera completa:

In [ ]:
steps_per_epoch = int(len(train_data.corpus) / batch_size)
steps_per_epoch

Entrenamos el modelo por únicamente 5 épocas para reducir el tiempo de entrenamiento:

**Nota**: La ejecución de la celda siguiente puede tomar 47 minutos con la GPU gratuita de Google Colab. Puede cambiar el número de épocas de entrenamiento para reducir el tiempo de ejecución.

In [ ]:
%%time
epochs = 5 # Cambie el número de épocas si lo desea

model = SimpleBERT(sp.vocab_size(), model_dim, block_size, blocks_num, heads_num)
model.to(device)
train_losses, val_losses = training_loop(model, lr=0.001, max_steps=epochs*steps_per_epoch, data_loader=train_data, train_eval=train_eval_data, val_eval=val_data, steps_per_log=steps_per_epoch//2)

In [ ]:
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses, label='Validation loss')
plt.legend()
plt.show()

Guarde y cargue nuevamente el modelo pre-entrenado para que no deba reentrenar el modelo en caso de reiniciar el notebook. Recuerde guardarlo en su memoria local en algún lugar permanente.

In [ ]:
# Guardar el modelo
torch.save(model.state_dict(), 'pretrained_BERT.pt')

In [ ]:
# Cargar el modelo
model = SimpleBERT(sp.vocab_size(), model_dim, block_size, blocks_num, heads_num)
model.load_state_dict(torch.load('pretrained_BERT.pt', weights_only=True))
model.to(device)

## 4. Fine tuning del modelo

En esta sección vamos a utilizar nuestro modelo pre-entrenado de BERT para una tarea de pregunta-respuesta. Con este fin, vamos a usar el dataset [SQuAD](https://huggingface.co/datasets/rajpurkar/squad), este dataset consiste en 100 mil ejemplos para los cuales se tiene una pregunta, un artículo de Wikipedia como contexto, y una respuesta (la cual se extrae del contexto).

## 4.1. Preparación de los datos

Descarguemos el dataset, y preparemos los datos para realizar el fine tuning.

In [ ]:
ds = load_dataset("rajpurkar/squad")

Exploremos el contenido del primer item en el dataset:

In [ ]:
sample = ds['train'][0]
sample

Como se observa, cada item del dataset contiene el título del artículo de Wikipedia, el contenido del artículo, la pregunta, y la respuesta a la pregunta.

Posteriormente, vamos a procesar cada ejemplo para obtener un formato que sea conveniente para el entrenamiento del modelo. En razón de lo cual vamos a codificar todo el input, vamos a agregar un token que no utilizamos en entrenamiento para representar la separación entre la pregunta y el contexto, y como labels vamos a utilizar los índices en los que empieza y termina la respuesta dentro del contexto.

In [ ]:
def format_example(example, block_size = 256):
    # Codificamos la pregunta y agregamos el token de separación (en este caso usamos el token de fin de secuencia que no fue usado en el entrenamiento)
    question_encoded = sp.encode(example['question']) + [sp.eos_id()]

    # Tokenizamos el contexto y la respuesta por token para buscar la respuesta en el contexto
    context_tokens = sp.encode_as_pieces(example['context'])
    answer_tokens = sp.encode_as_pieces(example['answers']['text'][0])

    # Inicializamos las variables para guardar la posición de inicio y fin de la respuesta
    answer_start = None
    answer_end = None

    # Creamos el input completo con la pregunta y el contexto que se le pasará al modelo
    complete_input = question_encoded + sp.encode(example['context'])

    # Buscamos la primera aparición de la respuesta en el contexto
    if len(answer_tokens) > 0:
        for i in range(0, len(context_tokens) - len(answer_tokens) + 1):
            passage = context_tokens[i:i+len(answer_tokens)]
            if passage == answer_tokens:
                answer_start = i
                answer_end = i + len(answer_tokens) - 1 # Se resta 1 porque si passage es context_tokens[0:10] el índice de la última palabra es 9
                break
        # Si no se encontró la respuesta en el contexto devolvemos [] como label
        # Esto puede pasar si los tokens de la respuesta no coinciden exactamente con los tokens del contexto, por ejemplo si hay comillas dobles en el contexto y no en la respuesta
        if answer_start is None:
            return {'input': complete_input, 'label': []}
    # Si la respuesta está vacía (no hay respuesta para la pregunta) devolvemos [] como label
    else:
        return {'input': complete_input, 'label': []}

    # Agregamos al inicio de la respuesta el número de tokens de la pregunta
    answer_start += len(question_encoded)
    answer_end += len(question_encoded)

    # Agregamos padding a los ejemplos con menos de block_size tokens
    complete_input = np.pad(complete_input, pad_width=(0, 0 if len(complete_input) > block_size else block_size - len(complete_input)), constant_values=sp.pad_id())

    return {'input': complete_input, 'label': [answer_start, answer_end]}

Revisemos un ejemplo del formato de nuestros datos de entrenamiento:

In [ ]:
%%time
example = format_example(ds['train'][1])
example

In [ ]:
# Dato de entrada del modelo decodificado
print(sp.decode(example['input'].tolist())) # Recuerde que el token de fin de secuencia no es visible en la decodificación
# Respuesta esperada
print('Respuesta esperada:', ds['train'][1]['answers']['text'][0])
# Respuesta esperada usando el índice de inicio y fin de la respuesta
print('Respuesta extraída de los datos:', sp.decode(example['input'][example['label'][0]:example['label'][1]+1].tolist()))

Ahora procesamos todos los datos de entrenamiento:

**Nota**: La ejecución de la celda siguiente puede tomar 2 minutos.

In [ ]:
train_data = ds['train'].map(format_example)

Vamos a ignorar los ejemplos para los cuales no hay respuesta o no encontramos respuesta:

In [ ]:
train_data = train_data.filter(lambda x: len(x['label']) > 0)
train_data

Vamos a conservar únicamente los ejemplos con 256 tokens o menos para estar alineados con el tamaño de secuencias que nuestro modelo vio durante su entrenamiento:

In [ ]:
train_data = train_data.filter(lambda x: len(x['input']) <= 256)
train_data

Finalmente, cambiamo el formato del Dataset para obtener tensores de Pytorch.

In [ ]:
train_data.set_format(type='torch')

Realizamos los mismos pasos de procesamiento con el set de validación:

In [ ]:
val_data = ds['validation'].map(format_example)

val_data = val_data.filter(lambda x: len(x['label']) > 0)

val_data = val_data.filter(lambda x: len(x['input']) <= 256)

val_data.set_format(type='torch')

val_data

### 4.2. Fine tuning del modelo

En esta subsección vamos a realizar el fine tuning de nuestro propio modelo BERT pre-entrenado para la tarea de pregunta-respuesta usando el dataset **SQuAD**.
En el paper original, se hace este mismo fine tuning en este dataset, y lo realizan de la siguiente forma: primero, por medio del modelo pre-entrenado se obtienen los embeddings contextuales para todos los tokens, luego se hace producto punto entre cada uno de los tokens y un token nuevo que vamos a llamar `start_ans_emb` (los pesos de este token deben ser entrenados durante el fine tuning), y finalmente el resultado del producto punto (puntaje de similitud) de todos los tokens se pasa por la Softmax para encontrar el token que mayor puntaje tiene, este será la predicción del token en el que inicia la respuesta. Para el token de fin de la respuesta, se hace lo mismo.

In [ ]:
class SQuADModel(nn.Module):
    def __init__(self, pretrained_model) -> None:
        super().__init__()
        self.pretrained_model = pretrained_model
        self.start_ans_emb = nn.Parameter(torch.randn(pretrained_model.E.embedding_dim, 1) / pretrained_model.E.embedding_dim**0.5)
        self.end_ans_emb = nn.Parameter(torch.randn(pretrained_model.E.embedding_dim, 1)  / pretrained_model.E.embedding_dim**0.5)

    def forward(self, x, **kwargs):
        self.pretrained_model(x)
        contextual_embeds = self.pretrained_model.contextual_embeddings

        # Producto punto entre los embeddings contextuales y los embeddings de inicio y fin de respuesta
        start_logits = (contextual_embeds @ self.start_ans_emb).squeeze() # [batch_size, block_size]
        end_logits = (contextual_embeds @ self.end_ans_emb).squeeze() # [batch_size, block_size]

        return {'start_logits': start_logits, 'end_logits': end_logits}

Veamos un ejemplo de los datos de entrada y salida del modelo al que vamos realizar el fine tuning:

In [ ]:
# Tomamos un bacth con dos ejemplos
sample_data = train_data.iter(2).__next__()
x, y = sample_data['input'], sample_data['label']

x.shape, y.shape

Tenemos dos ejemplos cada uno con 256 tokens que van a ser la entrada al modelo, y como salida, tenemos las posiciones de incicio y salida de la respuesta dentro de los 256 tokens para cada uno de los dos ejemplos.

Ahora, pasemos este mini-batch por nuestro nuevo modelo:

In [ ]:
finetuned_model = SQuADModel(model)

with torch.no_grad():
    x = x.to(device)
    finetuned_model.to(device)
    start_logits, end_logits = finetuned_model(x).values()
    print(start_logits.shape, end_logits.shape)

Tras pasar por el modelo, obtenemos para los dos ejemplos los logits, que corresponden a los puntajes de similitud asignados por nuestro modelo a cada uno de los tokens con respecto al token especial de inicio y fin de respuesta.

Este nuevo modelo contiene 1024 parámetros más que nuestro modelo inicial, debido a los dos nuevos embeddings agregados.

In [ ]:
print(f'Número de parametros del modelo: {sum([p.nelement() for p in finetuned_model.parameters()]):,}')

De la misma manera que lo hicimos con el modelo en pre-entrenamiento, vamos a crear una función para estimar la función de costo en el set de entrenamiento y validación. Para la tarea de encontrar las posiciones de inicio y fin de la respuesta, nuestra función de costo va a hacer la log cross-entropy por la posición de inicio y por la posición de fin, las cuales vamos a sumar para obtener la función de costo final.

In [ ]:
# Función para estimar el valor de pérdida en el set de entrenamiento, y en el de validación
@torch.no_grad
def estimate_SQuAD_loss(model: torch.nn.Module, train_dataset, val_dataset, batch_size: int = 32, eval_iters: int = 10):
    model.eval() # Se debe informar a Pytorch que estamos en modo evaluación, esta información puede ser usada por alguna de las capas usadas como por ejemplo Dropout
    train_losses = []
    val_losses = []
    # Iteramos sobre un número de batches definido para tener una mejor estimación del valor de pérdida en cada uno de los sets
    dataloader = iter(DataLoader(train_dataset.select_columns(['input', 'label']), batch_size=batch_size, shuffle=True))
    for i in range(eval_iters):
        batch = next(dataloader)
        x, y = batch['input'], batch['label']
        x, y = x.to(device), y.to(device)

        # Las siguientes líneas no son para el modelo actual, pero si para hacer el fine tuning del BERT original, que veremos más adelante
        attn_mask = None
        if 'attn_mask' in batch:
            attn_mask = torch.tensor(batch['attn_mask']).to(device)
        token_type_ids = None
        if 'token_type_ids' in batch:
            token_type_ids = torch.tensor(batch['token_type_ids']).to(device)

        # Se obtienen los logits para el inicio y fin de la respuesta
        start_logits, end_logits = model(x, token_type_ids=token_type_ids, attention_mask=attn_mask).values()
        start_loss = torch.nn.functional.cross_entropy(start_logits, y[:,0])
        end_loss = torch.nn.functional.cross_entropy(end_logits, y[:,1])
        # La pérdida total es la suma de las pérdidas de inicio y fin
        loss = start_loss + end_loss
        train_losses.append(loss.item())

    # Se realiza el mismo proceso en el set de validación
    dataloader = iter(DataLoader(val_dataset.select_columns(['input', 'label']), batch_size=batch_size, shuffle=True))
    for i in range(eval_iters):
        batch = next(dataloader)
        x, y = batch['input'], batch['label']
        x, y = x.to(device), y.to(device)

        attn_mask = None
        if 'attn_mask' in batch:
            attn_mask = torch.tensor(batch['attn_mask']).to(device)
        token_type_ids = None
        if 'token_type_ids' in batch:
            token_type_ids = torch.tensor(batch['token_type_ids']).to(device)

        start_logits, end_logits = model(x, token_type_ids=token_type_ids, attention_mask=attn_mask).values()
        start_loss = torch.nn.functional.cross_entropy(start_logits, y[:,0])
        end_loss = torch.nn.functional.cross_entropy(end_logits, y[:,1])
        loss = start_loss + end_loss
        val_losses.append(loss.item())
    train_loss = sum(train_losses)/eval_iters
    val_loss = sum(val_losses)/eval_iters

    model.train()
    return train_loss, val_loss

Ahora creamos la función con la que vamos a realizar el fine tuning del modelo:

In [ ]:
def SQuAD_training_loop(model, lr, train_data, val_data, batch_size = 32, epochs = 1, weight_decay: float = 0.0, log_steps = 10):
    train_losses = []
    val_losses = []

    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay, fused=True)

    print(f'Entrenando por {epochs} épocas')
    for epoch in range(epochs):
        dataloader = DataLoader(train_data.select_columns(['input', 'label']), batch_size=batch_size, shuffle=True)
        prev_step = 0
        t0 = time.time()
        for step, batch in enumerate(dataloader):
            optimizer.zero_grad()

            x, y = batch['input'], batch['label']
            x, y = x.to(device), y.to(device)

            # Las siguientes líneas no son para el modelo actual, pero si para hacer el fine tuning del BERT original, que veremos más adelante
            attn_mask = None
            if 'attn_mask' in batch:
                attn_mask = torch.tensor(batch['attn_mask']).to(device)
            token_type_ids = None
            if 'token_type_ids' in batch:
                token_type_ids = torch.tensor(batch['token_type_ids']).to(device)

            start_logits, end_logits = model(x, token_type_ids=token_type_ids, attention_mask=attn_mask).values()

            # Se obtienen los logits para el inicio y fin de la respuesta
            start_loss = torch.nn.functional.cross_entropy(start_logits, y[:,0])
            end_loss = torch.nn.functional.cross_entropy(end_logits, y[:,1])
            # La pérdida total es la suma de las pérdidas de inicio y fin
            loss = start_loss + end_loss

            loss.backward()
            optimizer.step()

            # Logs
            if (step+1) % log_steps == 0 or step == 0 or step == len(dataloader)-1:
                train_loss, valid_loss = estimate_SQuAD_loss(model, train_data, val_data, batch_size=batch_size, eval_iters=10)
                t1 = time.time()
                dt = (t1 - t0)
                t0 = time.time()
                examples_per_sec = batch_size*(step+1-prev_step)/dt
                prev_step = step+1
                print(f'Step {step+1:4d} - Train loss: {train_loss:4.2f} - Valid loss: {valid_loss:4.2f}, exs/sec:{examples_per_sec:5.0f}')
                train_losses.append(train_loss)
                val_losses.append(valid_loss)

    return train_losses, val_losses

Calculamos el número de batches necesarios para iterar completamente sobre el dataset de **SQuAD**.

In [ ]:
batch_size = 32
steps_per_batch = len(train_data) // batch_size
steps_per_batch

Realizamos solo 3 épocas de fine tuning:

**Nota**: La ejecución de la celda siguiente puede tomar 18 minutos con la GPU gratuita de Google Colab. Puede cambiar el número de épocas de entrenamiento para reducir el tiempo de ejecución.

In [ ]:
%%time
finetuned_model = SQuADModel(model)
finetuned_model.to(device)
train_losses, val_losses = SQuAD_training_loop(finetuned_model, 0.0001, train_data, val_data, batch_size=batch_size, log_steps=100, epochs=3)

In [ ]:
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses, label='Validation loss')
plt.legend()
plt.show()

Creamos una función para evaluar el desempeño del modelo en el set de validación. Vamos a medir el accuracy del modelo en la predicción del token en el que inicia la respuesta, y el token en el que finaliza por aparte:

In [ ]:
def eval_accuracy(model, dataset, batch_size = 128):
    model.eval()
    with torch.no_grad():
        start_match = 0
        end_match = 0
        examples = 0

        # Iteramos sobre todo el dataset en batches de tamaño batch_size
        for batch in dataset.iter(batch_size):
            # Las siguientes líneas no son para el modelo actual, pero si para hacer el fine tuning del BERT original, que veremos más adelante
            if 'token_type_ids' in batch and 'attn_mask' in batch:
                x, token_type_ids, attn_mask, y = batch['input'], batch['token_type_ids'], batch['attn_mask'], batch['label']
                x, token_type_ids, attn_mask = x.to(device), token_type_ids.to(device), attn_mask.to(device)
            # Se trasladan los tensores al dispositivo en el que se encuentra el modelo
            else:
                token_type_ids = None
                attn_mask = None
                x, y = batch['input'], batch['label']
                x = x.to(device)

            # Se obtienen los logits para el inicio y fin de la respuesta
            start_logits, end_logits = model(x, token_type_ids=token_type_ids, attention_mask=attn_mask).values()

            # Se obtienen los índices de los tokens con mayor probabilidad de ser el inicio y fin de la respuesta
            pred_start = torch.argmax(start_logits, dim=-1).cpu()
            pred_end = torch.argmax(end_logits, dim=-1).cpu()

            # Se revisa si los índices de los tokens predecidos coinciden con los índices de los labels
            ans_tokens_pos = y
            start_match += (pred_start == ans_tokens_pos[:,0]).sum().item()
            end_match += (pred_end == ans_tokens_pos[:,1]).sum().item()

            examples += x.shape[0]

        # Se calcula el accuracy para el inicio y fin de la respuesta
        print(start_match/examples, end_match/examples)

In [ ]:
%%time
eval_accuracy(finetuned_model, val_data)

El modelo logra acertar en una quinta parte de los ejemplos del set de validación (ejemplos nunca vistos durante el entrenamiento). Tenga en cuenta que estamos evaluando únicamente el accuracy de la predicción de la posición, y no estamos teniendo en cuenta las posibles intersecciones entre la respuesta predicha por el modelo y la respuesta original.

## 5. Fine tuning del modelo original de BERT pre-entrenado

En esta sección vamos a utilizar el modelo original de BERT ya pre-entrenado, el cual es un modelo de más de 100 millones de parámetros que podemos usar como punto de partida para hacer el fine tuning sobre el dataset de **SQuAD**.

In [ ]:
# Cargamos el tokenizador de BERT
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

Vamos a descargar el checjpoint de BERT pre-entrenado desde Hugging Face, y vamos a utilizar la cabeza adecuada para nuestra tarea (pregunta-respuesta), lo cual facilita el uso del modelo.

In [ ]:
model = BertForQuestionAnswering.from_pretrained(
    "bert-base-uncased", # Se debe usar el mismo modelo que el tokenizador
)

model.cuda()

El objetivo de BERT, es crear embedding contextuales para cada token que recojan la información relevante del contexto.

Veamos los embeddings generados por BERT para un mismo token usado en contextos diferentes:

In [ ]:
with torch.no_grad():
    # Creamos distintos contextos en los que la palabra 'bank' puede tener un significado distinto
    # Los primeros 3 ejemplos son contextos en los que 'bank' se refiere a la orilla de un río
    # Los siguientes 3 ejemplos son contextos en los que 'bank' se refiere a una institución financiera
    examples = [
        'The bank of the river is on the other side.',
        'The bank of the river was dirty.',
        'The bank of the canal was beautiful.',
        'The bank was closed today.',
        'The bank was robbed yesterday.',
        'The bank was full of people.',
    ]

    # Creamos una pequeña función auxiliar para obtener los embeddings contextuales de los tokens
    def get_contextual_embeddings(model, example):
        outputs = model(**example.to('cuda'), output_hidden_states=True)
        return outputs.hidden_states[-1][0].cpu().numpy()

    # Tokenizamos los ejemplos
    encoded_examples = [tokenizer(example, return_tensors='pt') for example in examples]

    # Obtenemos el índice de la palabra 'bank' en cada uno de los ejemplos
    bank_id = tokenizer.encode('bank', add_special_tokens=False)[0]
    bank_positions = [example['input_ids'][0].tolist().index(bank_id) for example in encoded_examples]

    # Obtenemos el embedding contextual de la palabra 'bank' en cada uno de los ejemplos
    embeddings = np.array([get_contextual_embeddings(model, example)[pos] for pos, example in zip(bank_positions, encoded_examples)])

    # Reducimos las dimensiones de los vectores de embeddings para poder visualizarlos
    tsne = TSNE(n_components=2, random_state=seed, perplexity=5)
    embeddings_2d = tsne.fit_transform(embeddings)

    # Graficamos los embeddings reducidos con el índice del ejemplo al que pertenece como etiqueta
    for i in range(len(examples)):
        plt.scatter(embeddings_2d[i,0], embeddings_2d[i,1], c='blue' if i <= 2 else 'red')
        # Add the text of the examples beside the point
        plt.annotate(examples[i], xy=(embeddings_2d[i]), xytext=embeddings_2d[i])

Si bien no hay una agrupación perfecta de la palabra "bank" según su significado, podemos observar como un mismo token tiene representaciones distintas según su contexto. Se espera que en un modelo entrenado el embedding contenga información suficiente sobre el token y el contexto en el que se encuentra para poder identificar diferencias en el significado del token, así como también otro tipo de información presente en el contexto que pueda agregar significado al token.

### 5.1. Procesamiento de los datos
Para utilizar el checkpoint pre-entrenado de BERT se debe usar el mismo tokenizador usado por BERT, además del modelo. Este se debe a que los tokens deben corresponder exactamente con los tokens usado en pre-entrenamiento, de modo que la matriz de embeddings coincida perfectamente, y obtengamos los embeddings contextuales correctos, por lo que demos procesar nuevamente los datos.

Verificamos que el tokenizador funcione correctamente. El argumento `return_offsets_mapping` nos devuelve las posiciones de la cadena de texto original que corresponden cada token. Por otro lado, con `add_special_tokens` podemos indicar que no queremos que se agreguen tokens especiales (como el token de inicio de secuencia) de manera automática.

In [ ]:
tokenizer('hello there', return_offsets_mapping=True, add_special_tokens=False)

In [ ]:
tokenizer.tokenize('This is some text. asds asd')

In [ ]:
tokenizer.encode('asdsd asd')

Debido a el cambio del tokenizador, debemos procesar nuevamente el dataset:

In [ ]:
def format_example(example):
    # Tokenizamos la pregunta y el contexto
    # El tokenizador de BERT ya agrega los tokens especiales [CLS] al inicio y [SEP] para separar la pregunta del contexto y al final del contexto
    # Por medio de los argumentos padding y max_length, el tokenizador se encarga de agregar padding a los ejemplos más cortos
    # El argumento return_tensors='np' indica que se devuelven tensores de numpy
    # Debido a que el contexto lo pasamos en el argumento text_pair, el tokenizador crea automáticamente los token_type_ids
    # En los token_type_ids, los tokens de la pregunta tienen un 0 y los tokens del contexto tienen un 1
    # Los token_type_ids los usa BERT para agregar un embedding especial a los tokens de la pregunta y del contexto, este embedding se entrenó en
    # el modelo original para la tarea de predecir si las secuencias de texto separadas por el token [SEP] son contiguas o no
    encoded = tokenizer(example['question'], text_pair=example['context'], return_offsets_mapping=True, return_tensors='np', padding='max_length', max_length=256)

    # Se crea una máscara para reconocer los caracteres que pertenecen a la respuesta antes de tokenizar
    answer_start = example['answers']['answer_start'][0]
    answer_length = len(example['answers']['text'][0])
    raw_mask = np.full(len(example['context']), 0)
    if answer_length > 0:
        raw_mask[answer_start:answer_start+answer_length] = 1

    # Se crea otra máscara para reconocer los tokens (despues de tokenizar) que contienen caracteres que hacen parte de la respuesta
    answers_mask = np.full_like(encoded.input_ids[0], 0)
    for i, (tok_type , (tok_start, tok_end)) in enumerate(zip(encoded.token_type_ids[0], encoded.offset_mapping[0])):
        # Se ignoran los tokens de la pregunta, y los tokens que no tienen un offset asociado (tokens especiales)
        if tok_type == 0 or (tok_start == 0 and tok_end == 0):
            continue
        # Se busca si algún caracter del token está en la respuesta
        answers_mask[i] = raw_mask[tok_start:tok_end].max()

    # Se obtienen los índices de los tokens que contienen la respuesta
    answers_tok_id = np.arange(len(encoded.input_ids[0]))[answers_mask == 1]

    # La attention_mask es 1 para los tokens que no son padding y 0 para los tokens que son padding, esto se hace para que BERT ignore los tokens de padding
    return {'input': encoded.input_ids[0], 'token_type_ids': encoded.token_type_ids[0], 'label':[answers_tok_id[0],answers_tok_id[-1]], 'attn_mask': encoded.attention_mask[0]}

**Nota**: La ejecución de la celda siguiente puede tomar 5 minutos en Google Colab.

In [ ]:
%%time
train_data = ds['train']

train_data = ds['train'].map(format_example)

train_data = train_data.filter(lambda x: len(x['input']) <= 256)

train_data = train_data.filter(lambda x: len(x['label']) > 0)

train_data.set_format(type='torch')

In [ ]:
%%time
val_data = ds['validation']

val_data = ds['validation'].map(format_example)

val_data = val_data.filter(lambda x: len(x['input']) <= 256)

val_data = val_data.filter(lambda x: len(x['label']) > 0)

val_data.set_format(type='torch')

Verificamos el correcto procesamiento de los datos:

In [ ]:
example = format_example(ds['train'][0])
# Dato de entrada del modelo decodificado
print(tokenizer.decode(example['input']))
# Respuesta esperada
print('Respuesta esperada:', ds['train'][0]['answers']['text'][0])
# Respuesta esperada usando el índice de inicio y fin de la respuesta
print('Respuesta extraída de los datos:', tokenizer.decode(np.array(example['input'])[example['label'][0]:example['label'][-1]+1]))

### 5.2. Fine tuning del modelo

Vamos a realizar el finetuning del modelo original pre entrenado BERT.

En el resumen del modelo, podemos evidenciar cómo la capa final (qa_outputs) es una capa final con dos salidas. Estas dos salidas son el equivalente al logit que indica el token de inicio, y al logit que indica el token de fin de la respuesta. Es decir, en vez de utilizar dos embeddings adicionales para representar los token de inicio y fin de respuesta, y luego hacer el producto punto por cada uno de los tokens, el enfoque en Hugging Faces es simplemente agregar una capa lineal con dos neuronas de salida, lo cual es equivalente a la estrategia utilizada en el paper original, a excepción de los parámetros extra correspondientes al bias.

Conozcamos el número de parámetros del modelo:

In [ ]:
print(f'Número de parametros del modelo: {sum([p.nelement() for p in model.parameters()]):,}')

Para evitar dañar los parámetros ya entrenados por el modelo original, y para visualizar de mejor manera el efecto que tiene reusar los embeddings contextuales calculados por el modelo original para la tarea de SQuAD, vamos a congelar los parámetros del BERT original, y solo vamos a entrenar los parámetros de la última capa lineal del modelo.

In [ ]:
for name, p in model.named_parameters():
    print(f'{name}: {p.nelement():,}')
    if 'qa_outputs' not in name:
        p.requires_grad = False

Es decir, que solo vamos a entrenar 1.538 parámetros:

In [ ]:
print(f'Parámetros entrenables: {sum([p.nelement() for p in model.parameters() if p.requires_grad]):,}')

Batches necesarios para recorrer todo el dataset

In [ ]:
batch_size = 32
steps_per_batch = len(train_data) // batch_size
steps_per_batch

**Nota**: La ejecución de la celda siguiente puede tomar 22 minutos con la GPU gratuita de Google Colab.

In [ ]:
%%time

train_losses, val_losses = SQuAD_training_loop(model, 0.0001, train_data, val_data, batch_size=batch_size, log_steps=100, epochs=1)

**Nota**: La ejecución de la celda siguiente puede tomar 2 minutos en Google Colab.

In [ ]:
%%time
eval_accuracy(model, val_data)

Con solo una época de entrenamiento se alcanza un resultado aceptable. Ahora vamos a probar el desempeño tras 2 épocas de entrenamiento descongelando todos los parámetros del modelo.

In [ ]:
# Se descongelan los parámetros congelados
for name, p in model.named_parameters():
    if 'qa_outputs' not in name:
        p.requires_grad = True

**Nota**: La ejecución de la celda siguiente puede tomar 2 horas con la GPU gratuita de Google Colab. Puede reducir a 1 las épocas de entrenamiento para reducir el tiempo de ejecución.

In [ ]:
%%time

train_losses, val_losses = SQuAD_training_loop(model, 5e-5, train_data, val_data, batch_size=batch_size, log_steps=100, epochs=2)

**Nota**: La ejecución de la celda siguiente puede tomar 2 minutos con la GPU gratuita de Google Colab.

In [ ]:
%%time
eval_accuracy(model, val_data, batch_size=128)

Con 2 épocas de entrenamiento con todos los parámetros la mejora en el desempeño en la tarea es sustancial.

Veamos la respuesta encontrada por nuestro modelo para los ejemplos del set de validación.

In [ ]:
sample_id = 0 # Cambie el id del ejemplo si desea ver otro ejemplo

with torch.no_grad():
    example = val_data[[sample_id]]
    print('Question:', example['question'][0])
    print('Context:', example['context'][0])
    print('Exptected answer:', example['answers'][0]['text'][0])
    x, token_type_ids, attn_mask = example['input'], example['token_type_ids'], example['attn_mask']
    x, token_type_ids, attn_mask = x.to(device), token_type_ids.to(device), attn_mask.to(device)
    start_logits, end_logits = model(x, token_type_ids=token_type_ids, attention_mask=attn_mask).values()
    start_logits = start_logits.cpu()[0]
    end_logits = end_logits.cpu()[0]
    pred_start = torch.argmax(start_logits)
    pred_end = torch.argmax(end_logits)
    pred_ans_tokens = example['input'][0, pred_start:pred_end+1]
    print('Model answer:', tokenizer.decode(pred_ans_tokens))

## 6. Conclusiones

El modelo BERT, es una alternativa al modelo de GPT para la creación de embeddings contextuales. Es especialmente relevante para tareas en las que sea pertinente aprovechar el contexto completo, como tareas de clasificación a nivel de token (por ejemplo POS tagging). En el paper original, se demostró como por medio de la tarea de lenguaje enmascarado, los embeddings contextuales resultantes mejoran el desempeño para todo tipo de tareas de PLN con tan solo 2 o 3 rondas de fine tuning en la tarea específica.

Adicionalmente, los modelos tipo BERT suelen ser mucho más pequeños que los modelos GPT, lo que los hace una excelente alternativa cuando no se tienen tantos recursos de cómputo.